# SAFE-EAR — MedGemma evaluation on a free GPU

Runs the **actual open medical LLM (MedGemma)** through the SAFE-EAR safety
harness and produces the numbers the companion paper (paper B) needs, on a
**free** GPU (Colab free-tier **T4 16 GB** or Kaggle **T4/P100** are enough for
`google/medgemma-4b-it`).

**What you do (about 5 minutes of clicks, then it runs itself):**
1. In Colab: `Runtime → Change runtime type → GPU` (T4). In Kaggle: enable a GPU accelerator.
2. Accept the MedGemma license **once** at <https://huggingface.co/google/medgemma-4b-it> (free, click-through).
3. Create a read token at <https://huggingface.co/settings/tokens> and paste it when the login cell asks.
4. `Runtime → Run all`.
5. Copy the JSON printed at the end (or download the file) and paste it back to Claude — Claude wires the real MedGemma numbers into the paper.

No patient data are used: the benchmark is entirely expert-authored / synthetic and lives in the repo.

## 1. Install dependencies

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate "huggingface_hub>=0.24"
# torch ships preinstalled on Colab/Kaggle GPU runtimes.
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → GPU)."

## 2. Log in to Hugging Face
You must have accepted the MedGemma license at
<https://huggingface.co/google/medgemma-4b-it> first, otherwise the download is 403.

In [ ]:
from huggingface_hub import login
login()  # paste a read token from https://huggingface.co/settings/tokens

## 3. Get the SAFE-EAR code + benchmark
If the repository is private, replace the URL with
`https://<YOUR_GITHUB_PAT>@github.com/leemgs/stars-audiology.git`.

In [ ]:
import os
if not os.path.isdir("stars-audiology"):
    !git clone --depth 1 https://github.com/leemgs/stars-audiology.git
%cd stars-audiology/code/src
!ls run_llm_eval.py llm_medgemma.py redflag_benchmark.py safety.py

## 4. Run the MedGemma evaluation through the SAFE-EAR harness
This scores MedGemma **and** the two rule-based references on the identical
benchmark, adapter, and metrics. Greedy (deterministic) decoding. On a free T4
the 37-note benchmark takes roughly a few minutes.

In [ ]:
!python run_llm_eval.py \
    --extractor rule_v1,rule_v2,medgemma \
    --model google/medgemma-4b-it \
    --out /content/llm_extractor_eval.json \
    --latex /content/table_llm_compare.tex

## 5. Show the result (copy this JSON back to Claude)

In [ ]:
import json
path = "/content/llm_extractor_eval.json"
if not os.path.exists(path):
    path = "llm_extractor_eval.json"  # Kaggle: cwd fallback
with open(path) as f:
    res = json.load(f)
print(json.dumps(res, indent=2))
print("\n=== one-line summary ===")
for name, r in res["extractors"].items():
    if r.get("status") == "ok":
        e = r["end_to_end"]; fdl = r["field_level"]
        print(f"{name}: e2e-recall={e['sensitivity']:.2f} "
              f"e2e-spec={e['specificity']:.2f} macroF1={fdl['macro']['f1']:.2f}")
    else:
        print(f"{name}: {r.get('status')} ({r.get('reason','')})")

In [ ]:
# Optional: download the files (Colab)
try:
    from google.colab import files
    files.download(path)
    if os.path.exists("/content/table_llm_compare.tex"):
        files.download("/content/table_llm_compare.tex")
except Exception as e:
    print("(download only works in Colab; on Kaggle use the output panel)", e)

---
**Next:** paste the JSON above back into the chat. Claude will insert the real
MedGemma row into Table `llm_compare`, update the Results/Discussion with the
measured recall/specificity, and finalize the paper — resolving the circularity
concern with an extractor-independent number.

**If you hit CUDA out-of-memory** (unlikely on T4 for the 4B model), re-run the
eval cell after loading in 4-bit:
```
!pip -q install bitsandbytes
```
then in `llm_medgemma.load_medgemma` pass `dtype='float16'`, or switch
`--model google/medgemma-4b-it` to the same (it already fits); the 27B text model
(`google/medgemma-27b-text-it`) needs an A100 and is not required.